In [3]:
import numpy as np
import time
from pynq import Overlay, allocate, PL

# 1. Load the overlay
ol = Overlay("dac_adc.bit")

In [4]:
dma = ol.axi_dma_0
dma_send = ol.axi_dma_0.sendchannel

In [56]:
#without caliberation code


import xrfdc
import time
import numpy as np
from pynq import Overlay, allocate

# --- INITIALIZATION ---
# (Assumes Overlay 'ol' is already loaded)
rfdc = xrfdc.RFdc(ol.ip_dict['usp_rf_data_converter_0'])
target_block = rfdc.dac_tiles[0].blocks[0]
dma_send = ol.axi_dma_0.sendchannel
dma_recv = ol.axi_dma_0.recvchannel

# Constants
DATA_SIZE_RECV = 8192 
CARRIER_FREQ = 100 

# TARGET_RMS: These are the average "strengths" of your 0.5V, 1.0V, 1.5V, and 2.0V signals.
# You will need to calibrate these 4 numbers based on your first run.
# Update your constants with the real values you just measured
TARGET_RMS = [900, 1800, 2700, 3600] 
RMS_TO_BITS = {900: "00", 1800: "01", 2700: "10", 3600: "11"}

# Allocate Buffers
input_buffer = allocate(shape=(1024,), dtype=np.uint16)
output_buffer = allocate(shape=(DATA_SIZE_RECV,), dtype=np.int16)

def update_nco(rf_block, nco_freq):
    mixer_cfg = rf_block.MixerSettings
    mixer_cfg['Freq'] = nco_freq
    rf_block.MixerSettings = mixer_cfg
    rf_block.UpdateEvent(xrfdc.EVENT_MIXER)

def analyze_amplitude_time_domain(data):
    """Calculates the strength of the signal using RMS (Root Mean Square)"""
    # 1. Convert to float and remove DC offset
    float_data = data.astype(np.float32) - np.mean(data)
    
    # 2. Calculate RMS: sqrt(mean(squares))
    # This represents the average power/amplitude of the sine wave
    rms_val = np.sqrt(np.mean(float_data**2))
    return rms_val

def play_and_analyze_ask(sequence):
    print(f"Starting Time-Domain ASK Sequence: {sequence}")
    decoded_string = ""
    
    # 1. Set Carrier Frequency Once
    update_nco(target_block, CARRIER_FREQ)
    
    for i in range(0, len(sequence), 2):
        pair = sequence[i:i+2]
        
        # --- TRANSMIT ---
        if pair == "00": amp = 8192    # 0.25V target
        elif pair == "01": amp = 16384 # 0.5V target
        elif pair == "10": amp = 24576 # 0.75V target
        elif pair == "11": amp = 32767 # 1.0V target
        else: continue
            
        input_buffer[:] = amp
        dma_send.transfer(input_buffer)
        
      #  time.sleep(0.1) # Stabilization
        
        # --- RECEIVE & DECODE ---
        dma_recv.transfer(output_buffer)
        dma_recv.wait()
        
        detected_rms = analyze_amplitude_time_domain(output_buffer)
        
        # Find the closest matching RMS level
        closest_rms = min(TARGET_RMS, key=lambda x: abs(x - detected_rms))
        detected_bits = RMS_TO_BITS[closest_rms]
        
        decoded_string += detected_bits
        print(f"TX: {pair} -> RX RMS: {detected_rms:.1f} -> Decoded: {detected_bits}")
        
        #time.sleep(3)

    # 3. SHUTDOWN
    input_buffer[:] = 0 
    dma_send.transfer(input_buffer) 
    print(f"\nFinal Decoded String: {decoded_string}")

# --- Run ---
try:
    user_seq = input("Enter binary sequence: ")
    play_and_analyze_ask(user_seq)
except KeyboardInterrupt:
    print("\nStopped.")

Enter binary sequence:  010101010101010111001100


Starting Time-Domain ASK Sequence: 010101010101010111001100
TX: 01 -> RX RMS: 1795.9 -> Decoded: 01
TX: 01 -> RX RMS: 1830.5 -> Decoded: 01
TX: 01 -> RX RMS: 1824.1 -> Decoded: 01
TX: 01 -> RX RMS: 1831.3 -> Decoded: 01
TX: 01 -> RX RMS: 1826.7 -> Decoded: 01
TX: 01 -> RX RMS: 1833.6 -> Decoded: 01
TX: 01 -> RX RMS: 1822.5 -> Decoded: 01
TX: 01 -> RX RMS: 1832.0 -> Decoded: 01
TX: 11 -> RX RMS: 3609.6 -> Decoded: 11
TX: 00 -> RX RMS: 1089.1 -> Decoded: 00
TX: 11 -> RX RMS: 3600.3 -> Decoded: 11
TX: 00 -> RX RMS: 1109.8 -> Decoded: 00

Final Decoded String: 010101010101010111001100


In [50]:
#with caliberation part 1

import xrfdc
import time
import numpy as np
from pynq import Overlay, allocate

ol = Overlay("dac_adc.bit")
rfdc = xrfdc.RFdc(ol.ip_dict['usp_rf_data_converter_0'])
target_block = rfdc.dac_tiles[0].blocks[0]
dma_send = ol.axi_dma_0.sendchannel
dma_recv = ol.axi_dma_0.recvchannel

# Buffers
input_buffer = allocate(shape=(1024,), dtype=np.uint16)
output_buffer = allocate(shape=(8192,), dtype=np.int16)

def update_nco(rf_block, nco_freq):
    mixer_cfg = rf_block.MixerSettings
    mixer_cfg['Freq'] = nco_freq
    rf_block.MixerSettings = mixer_cfg
    rf_block.UpdateEvent(xrfdc.EVENT_MIXER)

def analyze_amplitude_time_domain(data):
    float_data = data.astype(np.float32) - np.mean(data)
    return np.sqrt(np.mean(float_data**2))

In [78]:
#with caliberation part 2

CARRIER_FREQ = 800 # Change this to test the limit (e.g., 100, 400, 600)

print(f"--- Calibrating Channel at {CARRIER_FREQ} MHz ---")
update_nco(target_block, CARRIER_FREQ)

# Send Pilot Signal (Full Scale 11)
input_buffer[:] = 32767
dma_send.transfer(input_buffer)
time.sleep(0.2) 

# Measure
dma_recv.transfer(output_buffer)
dma_recv.wait()
max_rms = analyze_amplitude_time_domain(output_buffer)

# Save the 'Ruler' globally so Cell 3 can see it
my_targets = [max_rms * 0.25, max_rms * 0.50, max_rms * 0.75, max_rms]
print(f"Calibration Saved. Max RMS at this freq: {max_rms:.1f}")

--- Calibrating Channel at 800 MHz ---
Calibration Saved. Max RMS at this freq: 2172.1


In [54]:
#with caliberation part 3

user_seq = input("Enter binary sequence: ")
decoded_string = ""
bit_map = ["00", "01", "10", "11"]

for i in range(0, len(user_seq), 2):
    pair = user_seq[i:i+2]
    amp = 8192 if pair=="00" else 16384 if pair=="01" else 24576 if pair=="10" else 32767
    
    input_buffer[:] = amp
    dma_send.transfer(input_buffer)
    #time.sleep(0.1)
    
    dma_recv.transfer(output_buffer)
    dma_recv.wait()
    
    current_rms = analyze_amplitude_time_domain(output_buffer)
    idx = np.argmin([abs(current_rms - t) for t in my_targets])
    
    decoded_string += bit_map[idx]
    print(f"TX {pair} -> RX {bit_map[idx]} (RMS: {current_rms:.1f})")

print(f"\nFinal Result: {decoded_string}")

Enter binary sequence:  01010101010110110110101111100001101010100


TX 01 -> RX 01 (RMS: 1885.4)
TX 01 -> RX 01 (RMS: 1841.3)
TX 01 -> RX 01 (RMS: 1843.3)
TX 01 -> RX 01 (RMS: 1876.9)
TX 01 -> RX 01 (RMS: 1841.8)
TX 01 -> RX 01 (RMS: 1855.7)
TX 10 -> RX 10 (RMS: 2747.9)
TX 11 -> RX 11 (RMS: 3662.8)
TX 01 -> RX 01 (RMS: 1937.3)
TX 10 -> RX 10 (RMS: 2748.2)
TX 10 -> RX 10 (RMS: 2771.3)
TX 11 -> RX 11 (RMS: 3673.7)
TX 11 -> RX 11 (RMS: 3688.8)
TX 10 -> RX 10 (RMS: 2825.5)
TX 00 -> RX 00 (RMS: 1057.3)
TX 01 -> RX 01 (RMS: 1838.1)
TX 10 -> RX 10 (RMS: 2761.5)
TX 10 -> RX 10 (RMS: 2763.8)
TX 10 -> RX 10 (RMS: 2772.1)
TX 10 -> RX 10 (RMS: 2765.3)
TX 0 -> RX 11 (RMS: 3659.6)

Final Result: 010101010101101101101011111000011010101011


In [79]:
#bit rate calculation


import time

# 1. User Input
user_seq = input("Enter binary sequence: ")
num_bits = len(user_seq)

if num_bits % 2 != 0:
    print("Error: Please enter an even number of bits.")
else:
    # 2. Start Timing
    start_time = time.perf_counter()

    decoded_string = ""
    bit_map = ["00", "01", "10", "11"]

    print(f"--- Starting Transmission of {num_bits} bits ---")

    for i in range(0, num_bits, 2):
        pair = user_seq[i:i+2]
        
        # Select Amplitude
        if pair == "00": amp = 8192
        elif pair == "01": amp = 16384
        elif pair == "10": amp = 24576
        elif pair == "11": amp = 32767
        
        # DAC Send
        input_buffer[:] = amp
        dma_send.transfer(input_buffer)
        
        # ADC Receive
        dma_recv.transfer(output_buffer)
        dma_recv.wait()
        
        # Decode using RMS
        current_rms = analyze_amplitude_time_domain(output_buffer)
        
        # Logic to find the closest calibrated target
        idx = np.argmin([abs(current_rms - t) for t in my_targets])
        decoded_string += bit_map[idx]

    # 3. End Timing
    end_time = time.perf_counter()
    total_duration = end_time - start_time

    # 4. Calculate Accuracy
    # Compare original string to decoded string bit-by-bit
    correct_bits = sum(1 for a, b in zip(user_seq, decoded_string) if a == b)
    accuracy_pct = (correct_bits / num_bits) * 100

    # 5. Calculate Data Rate
    bps = num_bits / total_duration

    print("\n" + "="*45)
    print(f"RESULTS FOR CARRIER {CARRIER_FREQ} MHz")
    print("="*45)
    print(f"Original:         {user_seq}")
    print(f"Decoded:          {decoded_string}")
    print("-" * 45)
    print(f"Total Bits:       {num_bits}")
    print(f"Correct Bits:     {correct_bits}")
    print(f"Accuracy:         {accuracy_pct:.2f}%")
    print(f"Total Time:       {total_duration:.4f} s")
    print(f"Data Rate:        {bps:.2f} bps")
    print("="*45)

Enter binary sequence:  1010010100101010001010011010100101010010101001010100100101111010100001010010100100101001010100


--- Starting Transmission of 94 bits ---

RESULTS FOR CARRIER 800 MHz
Original:         1010010100101010001010011010100101010010101001010100100101111010100001010010100100101001010100
Decoded:          1010010100101010001010011010100101010010101001010100100101111010100001010010100100101001010100
---------------------------------------------
Total Bits:       94
Correct Bits:     94
Accuracy:         100.00%
Total Time:       0.0939 s
Data Rate:        1000.70 bps
